# Timetable solver

In [ ]:
import json
import os
from collections import Counter, defaultdict

from ortools.sat.python import cp_model
from groups import build_groups, session_demand

SIZE = "real"
TIME_LIMIT = 60
SOFT_LIMIT = 120

dept = json.load(open(f"out/department_{SIZE}.json"))
groups = build_groups(dept)
student_groups = [g for g in groups if g["kind"] == "group"]
group_by_id = {g["id"]: g for g in groups}
level_by_id = {l["id"]: l for l in dept["levels"]}
days = dept["calendar"]["days"]
hours = dept["rules"]["session_minutes"] / 60

In [ ]:
sessions = [{"id": f"s{i:03d}", "module": m, "kind": k, "group": g}
            for i, (m, k, g) in enumerate(session_demand(dept, groups))]
len(sessions)

### Professor assignment

In [ ]:
def candidates(s):
    return [(v["rank"], p["id"]) for p in dept["professors"] for v in p["voeux"]
            if v["module"] == s["module"] and s["kind"] in v["kinds"]]

am = cp_model.CpModel()
y, cost = {}, []
for s in sessions:
    for rank, p in candidates(s):
        y[s["id"], p] = am.NewBoolVar("")
        cost.append(rank * y[s["id"], p])
    am.AddExactlyOne(y[s["id"], p] for _, p in candidates(s))

for p in dept["professors"]:
    mine = [v for (_, pid), v in y.items() if pid == p["id"]]
    if mine:
        am.Add(sum(mine) <= round(p["max_hours"] / hours))

am.Minimize(sum(cost))
asolver = cp_model.CpSolver()
asolver.parameters.max_time_in_seconds = 10
print(asolver.StatusName(asolver.Solve(am)))

for s in sessions:
    for _, p in candidates(s):
        if asolver.Value(y[s["id"], p]):
            s["prof"] = p

### Model

In [ ]:
times = [(d, sl["id"]) for d in days for sl in dept["calendar"]["slots"]]
time_index = {ts: t for t, ts in enumerate(times)}
blocked = {(u["prof"], u["day"], u["slot"]) for u in dept["unavailability"]}
room_types = dept["rules"]["room_types_for"]

# only keep rooms/times a session can actually use
for s in sessions:
    g = group_by_id[s["group"]]
    s["rooms"] = [r["id"] for r in dept["rooms"]
                  if r["type"] in room_types[s["kind"]] and r["capacity"] >= g["size"]]
    s["times"] = [t for t, (d, sl) in enumerate(times)
                  if d in level_by_id[g["level"]]["days"] and (s["prof"], d, sl) not in blocked]

model = cp_model.CpModel()
x, when = {}, {}
by_session, by_room, by_prof, by_group = (defaultdict(list) for _ in range(4))

for s in sessions:
    for t in s["times"]:
        here = []
        for r in s["rooms"]:
            v = model.NewBoolVar("")
            x[s["id"], t, r] = v
            here.append(v)
            by_session[s["id"]].append(v)
            by_room[t, r].append(v)
        # when[s, t] = 1 if the session is at time t, whatever the room
        when[s["id"], t] = model.NewBoolVar("")
        model.Add(when[s["id"], t] == sum(here))
        by_prof[t, s["prof"]].append(when[s["id"], t])
        by_group[t, s["group"]].append(when[s["id"], t])

len(x)

In [ ]:
for s in sessions:
    model.AddExactlyOne(by_session[s["id"]])
for vs in list(by_room.values()) + list(by_prof.values()):
    model.AddAtMostOne(vs)

def group_vars(g, t):
    # a group is busy during its own sessions and its section's lectures
    return by_group[t, g["id"]] + by_group[t, g["parent"]]

for g in student_groups:
    for t in range(len(times)):
        if len(group_vars(g, t)) > 1:
            model.AddAtMostOne(group_vars(g, t))

max_group = dept["rules"]["max_sessions_per_day_group"]
max_prof = dept["rules"]["max_sessions_per_day_prof"]
for d in days:
    day_t = [t for t, (day, _) in enumerate(times) if day == d]
    for g in student_groups:
        vs = [v for t in day_t for v in group_vars(g, t)]
        if vs:
            model.Add(sum(vs) <= max_group)
    for p in dept["professors"]:
        vs = [v for t in day_t for v in by_prof[t, p["id"]]]
        if vs:
            model.Add(sum(vs) <= max_prof)

In [ ]:
W = {"gap": 5, "single_day": 4, "prof_pref": 3, "late_start": 2, "td_before_lecture": 2, "late": 1}
penalties = []

slots_per_day = len(dept["calendar"]["slots"])
last_slot = slots_per_day - 1

# professors' disliked (day, slot), "all" = every day / the whole day
dislikes = set()
for p in dept["professors"]:
    for d in p.get("dislikes", []):
        for day in (days if d["day"] == "all" else [d["day"]]):
            for sl in (range(slots_per_day) if d["slot"] == "all" else [d["slot"]]):
                dislikes.add((p["id"], day, sl))

prof_of = {s["id"]: s["prof"] for s in sessions}
for (sid, t), v in when.items():
    day, slot = times[t]
    if slot == last_slot:
        penalties.append(W["late"] * v)
    if (prof_of[sid], day, slot) in dislikes:
        penalties.append(W["prof_pref"] * v)

# td/tp shouldn't be before the lecture on the same day
for s in sessions:
    g = group_by_id[s["group"]]
    if s["kind"] == "lecture" or g["kind"] != "group":
        continue
    for l in sessions:
        if l["kind"] != "lecture" or l["group"] != g["parent"] or l["module"] != s["module"]:
            continue
        bad = model.NewBoolVar("")
        for d in days:
            for i in range(slots_per_day):
                a = when.get((s["id"], time_index[d, i]))
                later = [when[l["id"], time_index[d, j]] for j in range(i + 1, slots_per_day)
                         if (l["id"], time_index[d, j]) in when]
                if a is not None and later:
                    model.Add(bad >= a + sum(later) - 1)
        penalties.append(W["td_before_lecture"] * bad)

for g in student_groups:
    for d in level_by_id[g["level"]]["days"]:
        # occ[i] = 1 if the group has a class in slot i of that day
        occ = []
        for i in range(slots_per_day):
            o = model.NewBoolVar("")
            model.Add(o == sum(group_vars(g, time_index[d, i])))
            occ.append(o)
        n = len(occ)

        # gaps
        for i in range(1, n - 1):
            before, after, gap = model.NewBoolVar(""), model.NewBoolVar(""), model.NewBoolVar("")
            for j in range(i):
                model.Add(before >= occ[j])
            for j in range(i + 1, n):
                model.Add(after >= occ[j])
            model.Add(gap >= before + after - 1 - occ[i])
            penalties.append(W["gap"] * gap)

        busy = model.NewBoolVar("")
        for o in occ:
            model.Add(busy >= o)
        single, late_start = model.NewBoolVar(""), model.NewBoolVar("")
        model.Add(single >= 2 * busy - sum(occ))
        model.Add(late_start >= busy - occ[0])
        penalties += [W["single_day"] * single, W["late_start"] * late_start]

model.Minimize(sum(penalties))
len(penalties)

### Solve

In [ ]:
class Progress(cp_model.CpSolverSolutionCallback):
    def __init__(self):
        super().__init__()
        self.n = 0

    def on_solution_callback(self):
        self.n += 1
        print(f"{self.WallTime():6.1f} s  #{self.n}  score {self.ObjectiveValue():.0f}")

def run(limit):
    sv = cp_model.CpSolver()
    sv.parameters.max_time_in_seconds = limit
    sv.parameters.num_workers = os.cpu_count() or 1
    st = sv.Solve(model, Progress())
    print(sv.StatusName(st), round(sv.WallTime(), 1), "s")
    return sv, st

model.ClearHints()

# 1) hard rules only: find any valid timetable
model.ClearObjective()
solver, status = run(TIME_LIMIT)

# 2) all rules, starting from that timetable
if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    for v in x.values():
        model.AddHint(v, solver.Value(v))
model.Minimize(sum(penalties))
solver2, status2 = run(SOFT_LIMIT)
if status2 in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    solver, status = solver2, status2

print("score:", solver.ObjectiveValue(), "bound:", solver.BestObjectiveBound())

In [ ]:
session_by_id = {s["id"]: s for s in sessions}

def read_result(sv):
    res = []
    for (sid, t, r), v in x.items():
        if sv.Value(v):
            s = session_by_id[sid]
            day, slot = times[t]
            res.append({"id": sid, "module": s["module"], "kind": s["kind"], "group": s["group"],
                        "prof": s["prof"], "room": r, "day": day, "slot": slot, "status": "scheduled"})
    return res

result = read_result(solver) if status in (cp_model.OPTIMAL, cp_model.FEASIBLE) else []
with open(f"out/timetable_{SIZE}.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)
len(result)

### Checks

In [ ]:
def count_clashes(result):
    clashes = 0
    for key in ("room", "prof"):
        c = Counter((r["day"], r["slot"], r[key]) for r in result)
        clashes += sum(n - 1 for n in c.values() if n > 1)
    for g in student_groups:
        c = Counter((r["day"], r["slot"]) for r in result if r["group"] in (g["id"], g["parent"]))
        clashes += sum(n - 1 for n in c.values() if n > 1)
    return clashes

def validate(result):
    problems = []
    rooms = {r["id"]: r for r in dept["rooms"]}
    if sorted(r["id"] for r in result if r["status"] == "scheduled") != sorted(s["id"] for s in sessions):
        problems.append("missing or duplicated sessions")
    for r in result:
        g = group_by_id[r["group"]]
        room = rooms[r["room"]]
        if room["type"] not in room_types[r["kind"]] or room["capacity"] < g["size"]:
            problems.append(f"{r['id']}: bad room")
        if (r["prof"], r["day"], r["slot"]) in blocked:
            problems.append(f"{r['id']}: prof unavailable")
        if r["day"] not in level_by_id[g["level"]]["days"]:
            problems.append(f"{r['id']}: day not allowed")
    for d in days:
        for g in student_groups:
            if sum(r["day"] == d and r["group"] in (g["id"], g["parent"]) for r in result) > max_group:
                problems.append(f"{g['id']}: too many sessions on {d}")
        for p in dept["professors"]:
            if sum(r["day"] == d and r["prof"] == p["id"] for r in result) > max_prof:
                problems.append(f"{p['id']}: too many sessions on {d}")
    for p in dept["professors"]:
        if sum(r["prof"] == p["id"] for r in result) * hours > p["max_hours"]:
            problems.append(f"{p['id']}: over max hours")
    return problems

print("clashes:", count_clashes(result))
print("violations:", validate(result)[:10])

In [ ]:
def evaluate(result):
    rep = Counter()
    for g in student_groups:
        mine = [r for r in result if r["group"] in (g["id"], g["parent"])]
        for d in days:
            used = sorted(r["slot"] for r in mine if r["day"] == d)
            if used:
                rep["gaps"] += used[-1] - used[0] + 1 - len(used)
                rep["late_start"] += used[0]
                rep["single_days"] += len(used) == 1
        for r in mine:
            if r["kind"] == "lecture":
                continue
            lec = [l for l in mine if l["kind"] == "lecture" and l["module"] == r["module"]]
            if lec and r["day"] == lec[0]["day"] and r["slot"] < lec[0]["slot"]:
                rep["td_before_lecture"] += 1
    rep["late"] = sum(r["slot"] == last_slot for r in result)
    rep["prof_pref_broken"] = sum((r["prof"], r["day"], r["slot"]) in dislikes for r in result)
    return dict(rep)

evaluate(result)

### Hard rules

In [ ]:
def check(dept, timetable):
    RULES = ["missing", "duplicate", "room_type", "capacity", "not_qualified", "prof_unavailable",
             "day_not_allowed", "room_clash", "prof_clash", "student_clash",
             "group_day_limit", "prof_day_limit", "prof_hours"]
    v = {rule: [] for rule in RULES}
    gs = {g["id"]: g for g in build_groups(dept)}
    lv = {l["id"]: l for l in dept["levels"]}
    rm = {r["id"]: r for r in dept["rooms"]}
    pr = {p["id"]: p for p in dept["professors"]}
    ru = dept["rules"]
    blk = {(u["prof"], u["day"], u["slot"]) for u in dept["unavailability"]}
    qual = {(p["id"], w["module"], k) for p in dept["professors"] for w in p["voeux"] for k in w["kinds"]}

    def family(gid):
        out, g = {gid}, gs[gid]
        while g["parent"]:
            out.add(g["parent"])
            g = gs[g["parent"]]
        return out | {x["id"] for x in gs.values() if x["parent"] == gid}

    needed = Counter(session_demand(dept, list(gs.values())))
    placed = Counter((r["module"], r["kind"], r["group"]) for r in timetable)
    for k, n in needed.items():
        if placed[k] < n:
            v["missing"].append(f"{k}: {n - placed[k]} not placed")
    for k, n in placed.items():
        if n > needed.get(k, 0):
            v["duplicate"].append(f"{k}: placed {n}, needed {needed.get(k, 0)}")

    for r in timetable:
        g, room, lab = gs[r["group"]], rm.get(r["room"]), f"{r['id']} {r['module']} {r['kind']}"
        if room is None or room["type"] not in ru["room_types_for"][r["kind"]]:
            v["room_type"].append(f"{lab}: {r['room']} wrong type")
        if room is not None and room["capacity"] < g["size"]:
            v["capacity"].append(f"{lab}: {room['capacity']} seats < {g['size']}")
        if (r["prof"], r["module"], r["kind"]) not in qual:
            v["not_qualified"].append(f"{lab}: {r['prof']} didn't ask for it")
        if (r["prof"], r["day"], r["slot"]) in blk:
            v["prof_unavailable"].append(f"{lab}: {r['prof']} blocked {r['day']} {r['slot']}")
        if r["day"] not in dept["calendar"]["days"] or r["day"] not in lv[g["level"]]["days"]:
            v["day_not_allowed"].append(f"{lab}: {r['day']}")

    for key, rule in (("room", "room_clash"), ("prof", "prof_clash")):
        at = defaultdict(list)
        for r in timetable:
            at[(r["day"], r["slot"], r[key])].append(r["id"])
        v[rule] += [f"{w} {d} {s}: {ids}" for (d, s, w), ids in at.items() if len(ids) > 1]

    at = defaultdict(list)
    for r in timetable:
        at[(r["day"], r["slot"])].append(r)
    for (d, s), rs in at.items():
        for i, a in enumerate(rs):
            v["student_clash"] += [f"{a['group']} + {b['group']} {d} {s}" for b in rs[i + 1:] if b["group"] in family(a["group"])]

    for g in gs.values():
        if g["kind"] == "group":
            for d, n in Counter(r["day"] for r in timetable if r["group"] in (g["id"], g["parent"])).items():
                if n > ru["max_sessions_per_day_group"]:
                    v["group_day_limit"].append(f"{g['id']}: {n} on {d}")
    for (p, d), n in Counter((r["prof"], r["day"]) for r in timetable).items():
        if n > ru["max_sessions_per_day_prof"]:
            v["prof_day_limit"].append(f"{p}: {n} on {d}")
    for p, n in Counter(r["prof"] for r in timetable).items():
        if n * hours > pr[p]["max_hours"]:
            v["prof_hours"].append(f"{p}: {n * hours} h > {pr[p]['max_hours']} h")
    return v

v = check(dept, [r for r in result if r["status"] == "scheduled"])
for rule, msgs in v.items():
    print(f"{rule:18s} {len(msgs):4d}   {msgs[:2] if msgs else ''}")
print("\nALL HARD RULES OK" if not any(v.values()) else "\nSOME RULES BROKEN")

### Excel export

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side

colour = {"lecture": "FFF2CC", "td": "DDEBF7", "tp": "E2EFDA"}
kind_fr = {"lecture": "Cours", "td": "TD", "tp": "TP"}
module_name = {m["id"]: m["name"] for m in dept["modules"]}
prof_name = {p["id"]: p["name"] for p in dept["professors"]}
room_name = {r["id"]: r["name"] for r in dept["rooms"]}
slots = dept["calendar"]["slots"]
line = Side(style="thin", color="999999")
box = Border(left=line, right=line, top=line, bottom=line)

def fill_sheet(ws, g):
    ws.append([g["name"]] + [f"{s['start']}-{s['end']}" for s in slots])
    mine = [r for r in result if r["status"] == "scheduled" and r["group"] in (g["id"], g["parent"])]
    for d in days:
        ws.append([d])
        for col, s in enumerate(slots, start=2):
            here = [r for r in mine if r["day"] == d and r["slot"] == s["id"]]
            if here:
                c = ws.cell(ws.max_row, col)
                c.value = "\n".join(f"{kind_fr[r['kind']]} {module_name[r['module']]}\n"
                                    f"{room_name[r['room']]} · {prof_name[r['prof']]}" for r in here)
                c.fill = PatternFill("solid", fgColor=colour[here[0]["kind"]])
    for row in ws.iter_rows():
        for c in row:
            c.border = box
            c.alignment = Alignment(wrap_text=True, vertical="center", horizontal="center")
    for c in ws[1] + tuple(r[0] for r in ws.iter_rows(min_row=2)):
        c.font = Font(bold=True)
    ws.column_dimensions["A"].width = 16
    for col in range(2, len(slots) + 2):
        ws.column_dimensions[ws.cell(1, col).column_letter].width = 28
    for r in range(2, ws.max_row + 1):
        ws.row_dimensions[r].height = 60

folder = f"out/timetables_{SIZE}"
os.makedirs(folder, exist_ok=True)
everything = Workbook()
everything.remove(everything.active)
for g in student_groups:
    name = g["name"].replace(" · ", " ")[:31]
    fill_sheet(everything.create_sheet(name), g)
    one = Workbook()
    one.active.title = name
    fill_sheet(one.active, g)
    one.save(f"{folder}/{name}.xlsx")
everything.save(f"out/timetable_{SIZE}.xlsx")
print(f"saved out/timetable_{SIZE}.xlsx (all groups) + {len(student_groups)} files in {folder}/")

### Slot finder

In [ ]:
def find_slots(prof_id, group_id, kind="td", top=5):
    g = group_by_id[group_id]
    if kind == "lecture" and g["kind"] == "group":
        g = group_by_id[g["parent"]]          # a lecture is for the whole section
    level = level_by_id[g["level"]]
    family = {g["id"], g["parent"]} | {x["id"] for x in groups if x["parent"] == g["id"]}
    students = [x for x in student_groups if x["id"] in family]
    active = [r for r in result if r["status"] != "cancelled"]

    busy_room = {(r["day"], r["slot"], r["room"]) for r in active}
    busy_prof = {(r["day"], r["slot"]) for r in active if r["prof"] == prof_id}
    busy_group = {(r["day"], r["slot"]) for r in active if r["group"] in family}

    options = []
    for d, sl in times:
        if d not in level["days"] or (prof_id, d, sl) in blocked:
            continue
        if (d, sl) in busy_prof or (d, sl) in busy_group:
            continue
        # every group concerned must stay under the daily limit
        if any(sum(r["day"] == d and r["group"] in (x["id"], x["parent"]) for r in active) >= max_group
               for x in students):
            continue
        group_day = sorted(s for (day, s) in busy_group if day == d)
        for room in dept["rooms"]:
            if room["type"] not in room_types[kind] or room["capacity"] < g["size"]:
                continue
            if (d, sl, room["id"]) in busy_room:
                continue
            score = 0
            if group_day and (sl == group_day[0] - 1 or sl == group_day[-1] + 1):
                score += 3              # right next to the group's other classes
            elif group_day:
                score -= 3              # would leave a gap
            else:
                score -= 2              # a day with only this session
            score -= (room["capacity"] - g["size"]) / 20
            if (prof_id, d, sl) in dislikes:
                score -= 3
            options.append((round(score, 2), d, sl, room["id"]))

    options.sort(key=lambda o: -o[0])
    start_time = {s["id"]: s["start"] for s in dept["calendar"]["slots"]}
    return [{"group": g["id"], "day": d, "time": start_time[sl], "slot": sl, "room": r, "score": sc}
            for sc, d, sl, r in options[:top]]

def claim(option, module_id, prof_id, kind="td"):
    result.append({"id": f"x{len(result):03d}", "module": module_id, "kind": kind, "group": option["group"],
                   "prof": prof_id, "room": option["room"], "day": option["day"],
                   "slot": option["slot"], "status": "extra"})

# examples: an extra TD, an extra TP and an extra lecture
for kind in ("td", "tp", "lecture"):
    ex = next(s for s in sessions if s["kind"] == kind)
    print(kind, ex["group"], find_slots(ex["prof"], ex["group"], kind, top=3))